# Modelling Travel Times using r5py

This script shows how I generate median travel times for each MSOA to their respective city centres for each scenario.  

Be prepared, this can take more than 24 hours to complete...  

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import r5py
import datetime
import os
from datetime import timedelta
from pathlib import Path
from functools import reduce
import time

notebook_start = time.perf_counter()

def elapsed(start):
    return str(timedelta(seconds=round(time.perf_counter() - start)))

ROOT = Path('../Data')
ROOT.resolve()

# setting links for easy reference later
post_processed = ROOT/'Postprocessed'
baseline_gtfs = ROOT/'Scenario 0_Baseline'
scheduled_gtfs = ROOT/'Scheduled'
reduced_gtfs = ROOT/'Scenario 1_Reduced Journeys'
frequency_gtfs = ROOT/'Scenario 2_Increased Frequencies'
compounded_gtfs = ROOT/'Scenario 3_Compounded'
results = ROOT/'Spatial Interaction Modelling'

# creating empty directory to store the outputs
os.makedirs(results, exist_ok=True)

# change accordingly
target_cities = ['Bristol', 'Leeds', 'Manchester']

target_dates = [
    '20260603', '20260610',
    '20260617', '20260624'
]

In [ ]:
# loading relevant datasets
orig_pt = gpd.read_file(post_processed/'origin_points.gpkg').to_crs(4326)
dest_pt = gpd.read_file(post_processed/'destination_points.gpkg').to_crs(4326)

In [ ]:
# MODELLING ALL SCHEDULED TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        [
            scheduled_gtfs/f'{city}_20260603_sched.zip',
            scheduled_gtfs/f'{city}_20260610_sched.zip',
            scheduled_gtfs/f'{city}_20260617_sched.zip',
            scheduled_gtfs/f'{city}_20260624_sched.zip',
            scheduled_gtfs/'railgtfs_cleaned.zip'
        ]
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        matrix['date'] = date

        city_results.append(matrix)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_scheduled_tt.csv', index=False)
    print(
        f"{city}'s scheduled travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL BASELINE TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        baseline_gtfs/city/f'{city}_20260603.zip',
        baseline_gtfs/city/f'{city}_20260610.zip',
        baseline_gtfs/city/f'{city}_20260617.zip',
        baseline_gtfs/city/f'{city}_20260624.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s0_tt.csv', index=False)
    print(
        f"{city}'s retrospective travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL REDUCED JOURNEY TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        reduced_gtfs/f'{city}_20260603_reduced.zip',
        reduced_gtfs/f'{city}_20260610_reduced.zip',
        reduced_gtfs/f'{city}_20260617_reduced.zip',
        reduced_gtfs/f'{city}_20260624_reduced.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s1_tt.csv', index=False)
    print(
        f"{city}'s reduced journey travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL REDUCED JOURNEY (INBOUND ONLY) TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        reduced_gtfs/f'{city}_20260603_inbound_reduced.zip',
        reduced_gtfs/f'{city}_20260610_inbound_reduced.zip',
        reduced_gtfs/f'{city}_20260617_inbound_reduced.zip',
        reduced_gtfs/f'{city}_20260624_inbound_reduced.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s1_inbound_tt.csv', index=False)
    print(
        f"{city}'s reduced journey (inbound only) travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL REDUCED JOURNEY (DEPRIVED ONLY) TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        reduced_gtfs/f'{city}_20260603_deprived_reduced.zip',
        reduced_gtfs/f'{city}_20260610_deprived_reduced.zip',
        reduced_gtfs/f'{city}_20260617_deprived_reduced.zip',
        reduced_gtfs/f'{city}_20260624_deprived_reduced.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s1_deprived_tt.csv', index=False)
    print(
        f"{city}'s reduced journey (deprived only) travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL INCREASED FREQUENCY TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        frequency_gtfs/f'{city}_20260603_frequent.zip',
        frequency_gtfs/f'{city}_20260610_frequent.zip',
        frequency_gtfs/f'{city}_20260617_frequent.zip',
        frequency_gtfs/f'{city}_20260624_frequent.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s2_tt.csv', index=False)
    print(
        f"{city}'s increased frequency travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL INCREASED FREQUENCY (INBOUND ONLY) TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        frequency_gtfs/f'{city}_20260603_inbound_frequent.zip',
        frequency_gtfs/f'{city}_20260610_inbound_frequent.zip',
        frequency_gtfs/f'{city}_20260617_inbound_frequent.zip',
        frequency_gtfs/f'{city}_20260624_inbound_frequent.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s2_inbound_tt.csv', index=False)
    print(
        f"{city}'s increased frequency (inbound only) travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL INCREASED FREQUENCY (DEPRIVED ONLY) TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        frequency_gtfs/f'{city}_20260603_deprived_frequent.zip',
        frequency_gtfs/f'{city}_20260610_deprived_frequent.zip',
        frequency_gtfs/f'{city}_20260617_deprived_frequent.zip',
        frequency_gtfs/f'{city}_20260624_deprived_frequent.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s2_deprived_tt.csv', index=False)
    print(
        f"{city}'s increased frequency (deprived only) travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL COMPOUNDED TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        compounded_gtfs/f'{city}_20260603_compounded.zip',
        compounded_gtfs/f'{city}_20260610_compounded.zip',
        compounded_gtfs/f'{city}_20260617_compounded.zip',
        compounded_gtfs/f'{city}_20260624_compounded.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s3_tt.csv', index=False)
    print(
        f"{city}'s compounded travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL COMPOUNDED (INBOUND ONLY) TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        compounded_gtfs/f'{city}_20260603_inbound_compounded.zip',
        compounded_gtfs/f'{city}_20260610_inbound_compounded.zip',
        compounded_gtfs/f'{city}_20260617_inbound_compounded.zip',
        compounded_gtfs/f'{city}_20260624_inbound_compounded.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s3_inbound_tt.csv', index=False)
    print(
        f"{city}'s compounded (inbound only) travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
# MODELLING ALL COMPOUNDED (DEPRIVED ONLY) TRAVEL TIMES
for city in target_cities:

    city_start = time.perf_counter()          # <-- add this

    gtfs_files = [
        compounded_gtfs/f'{city}_20260603_deprived_compounded.zip',
        compounded_gtfs/f'{city}_20260610_deprived_compounded.zip',
        compounded_gtfs/f'{city}_20260617_deprived_compounded.zip',
        compounded_gtfs/f'{city}_20260624_deprived_compounded.zip',
        scheduled_gtfs/'railgtfs_cleaned.zip'
    ]

    if city == 'Manchester':
        gtfs_files.extend([
            scheduled_gtfs/'Manchester_20260603_tram.zip',
            scheduled_gtfs/'Manchester_20260610_tram.zip',
            scheduled_gtfs/'Manchester_20260617_tram.zip',
            scheduled_gtfs/'Manchester_20260624_tram.zip'
        ])

    network = r5py.TransportNetwork(
        post_processed/f'{city}_ttwa.osm.pbf',
        gtfs_files
    )

    scoped_orig = orig_pt[orig_pt['TTWA11NM'] == city].copy()
    scoped_orig['id'] = scoped_orig['LSOA21CD']
    scoped_dest = dest_pt[dest_pt['TTWA11NM'] == city].copy()
    scoped_dest['id'] = scoped_dest['LSOA21CD']

    city_results = []

    for date in target_dates:

        matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=1,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt'}
        )

        mult_matrix = r5py.TravelTimeMatrix(
            network,
            origins=scoped_orig,
            destinations=scoped_dest,
            departure=datetime.datetime.strptime(
                f"{date} 07:00",
                "%Y%m%d %H:%M"
            ),
            departure_time_window=timedelta(minutes=180),
            transport_modes=[
                r5py.TransportMode.TRANSIT,
                r5py.TransportMode.WALK
            ],
            snap_to_network=True,
            max_public_transport_rides=2,
            max_time=timedelta(minutes=180)
        ).rename(
            columns={'travel_time': 'tt_mult'}
        )

        results_perday = matrix[['from_id', 'to_id', 'tt']].merge(
            mult_matrix[['from_id', 'to_id', 'tt_mult']],
            on=['from_id', 'to_id']
        ).merge(
            scoped_orig[['id', 'msoa21cd']],
            left_on='from_id',
            right_on='id',
            how='left'
        )

        results_perday['date'] = date

        city_results.append(results_perday)

    city_results = pd.concat(city_results, ignore_index=True)
    city_results.to_csv(results/f'{city}_s3_deprived_tt.csv', index=False)
    print(
        f"{city}'s compounded (deprived only) travel time modelling is complete! "
        f"({elapsed(city_start)} for this city, {elapsed(notebook_start)} total so far)"
    )

In [ ]:
print(f"Notebook finished in {elapsed(notebook_start)}")